# HDB project: data cleaning, step by step

This notebook walks through how the raw data.gov.sg files are cleaned. You see the data **before and after every step**, and the code behind each step.

It uses the code in the `hdb_pipeline` folder, so there is only one copy of the cleaning code. If you change a file in `hdb_pipeline`, rerun the notebook and you will see the change here.

**Before you start**
1. Save this notebook in the `hdb_project` folder, next to the `hdb_pipeline` folder.
2. In the first code cell, set `RAW` to your `data sets` folder (the one holding the original downloads).
3. Run the cells from top to bottom (Shift+Enter).

**Contents**
1. Setup
2. Look at the raw data
3. The generic cleaning steps, one at a time (resale data)
4. Resale-specific steps
5. Other datasets: childcare, MRT exits, parks
6. Run the whole pipeline in one go
7. Read the reports
8. (Optional) Geocode block coordinates

## 1. Setup

In [1]:
# >>> CHANGE THIS to the folder with the ORIGINAL data files (it is only ever read, never changed)
RAW = r"data sets"

# where all results go (created if missing, regenerated on every run)
OUT = r"pipeline_output"

In [2]:
from pathlib import Path
import inspect
import pandas as pd
from IPython.display import Markdown, display

from hdb_pipeline import config, io_utils, steps
from hdb_pipeline.datasets import REGISTRY
from hdb_pipeline.datasets.resale import ResaleCleaner
from hdb_pipeline.datasets.childcare import ChildcareCleaner
from hdb_pipeline.datasets.amenities import MrtExitCleaner, ParkCleaner
from hdb_pipeline.base import DatasetCleaner

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

RAW, OUT = Path(RAW), Path(OUT)
assert RAW.is_dir(), f"Folder not found: {RAW.resolve()}  ->  fix RAW in the cell above"

# unzip any .zip from RAW into OUT/_staging (RAW itself is never touched)
STAGING = OUT / config.STAGING_DIR
STAGING.mkdir(parents=True, exist_ok=True)
io_utils.expand_zips(RAW, STAGING)
FOLDERS = [RAW, STAGING]

def show_code(obj):
    """Print the source code of a function or class, nicely formatted."""
    display(Markdown(f"```python\n{inspect.getsource(obj)}\n```"))

# which file does each cleaner pick up?
for cls in REGISTRY:
    files = io_utils.find_files(FOLDERS, cls.patterns)
    print(f"{cls.__name__:22s} -> {len(files)} file(s): {[f.name for f in files]}")

ResaleCleaner          -> 5 file(s): ['Resale Flat Prices (Based on Approval Date), 1990 - 1999.csv', 'Resale Flat Prices (Based on Approval Date), 2000 - Feb 2012.csv', 'Resale Flat Prices (Based on Registration Date), From Jan 2015 to Dec 2016.csv', 'Resale Flat Prices (Based on Registration Date), From Mar 2012 to Dec 2014.csv', 'Resale flat prices based on registration date from Jan-2017 onwards.csv']
PropertyInfoCleaner    -> 1 file(s): ['HDBPropertyInformation.csv']
ChildcareCleaner       -> 1 file(s): ['ListingofCentres.csv']
HawkerCleaner          -> 1 file(s): ['HawkerCentresGEOJSON.geojson']
BusStopCleaner         -> 1 file(s): ['LTABusStop.geojson']
MrtExitCleaner         -> 1 file(s): ['LTAMRTStationExitGEOJSON.geojson']
ParkCleaner            -> 1 file(s): ['NParksParksandNatureReserves.geojson']
StreetCoordsCleaner    -> 0 file(s): []


## 2. Look at the raw data

The 5 resale files are read and stacked into one table. Everything is read **as text** first, so block numbers like `85A` and postal codes like `018956` keep their exact form.

In [3]:
resale_files = io_utils.find_files(FOLDERS, ResaleCleaner.patterns)
cleaner = ResaleCleaner(resale_files)
raw = cleaner.read()
print(f"{len(raw):,} rows from {len(resale_files)} files")
print(raw["source_file"].value_counts().to_string())
raw.head()

987,234 rows from 5 files
source_file
Resale Flat Prices (Based on Approval Date), 2000 - Feb 2012.csv                  369651
Resale Flat Prices (Based on Approval Date), 1990 - 1999.csv                      287196
Resale flat prices based on registration date from Jan-2017 onwards.csv           241031
Resale Flat Prices (Based on Registration Date), From Mar 2012 to Dec 2014.csv     52203
Resale Flat Prices (Based on Registration Date), From Jan 2015 to Dec 2016.csv     37153


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,source_file,remaining_lease,date_basis
0,1990-01,ANG MO KIO,1 ROOM,309,ANG MO KIO AVE 1,10 TO 12,31,IMPROVED,1977,9000,"Resale Flat Prices (Based on Approval Date), 1...",NaN,approval
1,1990-01,ANG MO KIO,1 ROOM,309,ANG MO KIO AVE 1,04 TO 06,31,IMPROVED,1977,6000,"Resale Flat Prices (Based on Approval Date), 1...",NaN,approval
2,1990-01,ANG MO KIO,1 ROOM,309,ANG MO KIO AVE 1,10 TO 12,31,IMPROVED,1977,8000,"Resale Flat Prices (Based on Approval Date), 1...",NaN,approval
3,1990-01,ANG MO KIO,1 ROOM,309,ANG MO KIO AVE 1,07 TO 09,31,IMPROVED,1977,6000,"Resale Flat Prices (Based on Approval Date), 1...",NaN,approval
4,1990-01,ANG MO KIO,3 ROOM,216,ANG MO KIO AVE 1,04 TO 06,73,NEW GENERATION,1976,47200,"Resale Flat Prices (Based on Approval Date), 1...",NaN,approval


In [4]:
# The files don't all have the same columns: only the newer files have remaining_lease
print("rows with remaining_lease filled:", (raw["remaining_lease"].fillna("") != "").sum())
raw.dtypes

rows with remaining_lease filled: 278184


month                  str
town                   str
flat_type              str
block                  str
street_name            str
storey_range           str
floor_area_sqm         str
flat_model             str
lease_commence_date    str
resale_price           str
source_file            str
remaining_lease        str
date_basis             str
dtype: object

## 3. The generic cleaning steps, one at a time

Every dataset goes through these steps, in this order. Here we run them on the resale data **one by one** to see what each does. Normally `DatasetCleaner.generic_steps()` runs them all for you.

In [5]:
show_code(DatasetCleaner.generic_steps)

```python
    def generic_steps(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.drop(columns=[c for c in df.columns if c.lower() in config.DROP_METADATA_COLUMNS])
        df = steps.strip_text(df)
        df = steps.mark_missing(df)
        df, empty = steps.drop_empty_columns(df)
        self.report["empty_columns_dropped"] = len(empty)
        df = steps.convert_types(df, skip=self.skip_type_conversion)
        subset = self.duplicate_subset or [c for c in df.columns if c != "source_file"]
        df, n = steps.handle_duplicates(df, self.duplicates, subset)
        self.report["duplicate_mode"] = self.duplicates
        self.report["duplicates_found"] = n
        df, n = steps.fill_numeric(df, self.fill)
        self.report["values_filled"] = n
        return df

```

### 3a. Strip stray spaces

In [6]:
show_code(steps.strip_text)

```python
def strip_text(df: pd.DataFrame) -> pd.DataFrame:
    """Remove leading/trailing spaces in every text cell (numbers are left alone)."""
    for col in df.columns:
        s = df[col]
        if not is_text(s):
            continue
        if _all_str(s):
            df[col] = s.str.strip()
        else:
            df[col] = s.map(lambda v: v.strip() if isinstance(v, str) else v)
    return df

```

In [7]:
df = raw.copy()
before = (df["street_name"] != df["street_name"].str.strip()).sum()
df = steps.strip_text(df)
after = (df["street_name"] != df["street_name"].str.strip()).sum()
print(f"street names with extra spaces: before {before}, after {after}")

street names with extra spaces: before 0, after 0


The resale files happen to have no stray spaces, so nothing changes here. The step is still worth keeping because it protects against future downloads, and other files (like the GeoJSON properties) do have them.

### 3b. Turn placeholder text into real missing values

In [8]:
show_code(steps.mark_missing)

```python
def mark_missing(df: pd.DataFrame) -> pd.DataFrame:
    """Turn placeholder text ('na', '-', ...) and all-zero IDs ('00000000') into real missing values."""
    tokens = set(config.MISSING_TOKENS)
    for col in df.columns:
        s = df[col]
        if not is_text(s):
            continue
        text = s.map(lambda v: v if isinstance(v, str) else None)
        missing = text.str.lower().isin(tokens).fillna(False).astype(bool)
        if is_id(col):
            missing |= text.str.fullmatch(r"0+").fillna(False).astype(bool)
        df[col] = s.mask(missing)
    return df

```

In [9]:
missing_before = df.isna().sum()
df = steps.mark_missing(df)
missing_after = df.isna().sum()
pd.DataFrame({"missing before": missing_before, "missing after": missing_after})

,missing before,missing after
month,0,0
town,0,0
flat_type,0,0
block,0,0
street_name,0,0
storey_range,0,0
floor_area_sqm,0,0
flat_model,0,0
lease_commence_date,0,0
resale_price,0,0


Nothing changes for the resale data: it contains no placeholder text. The 709,050 missing `remaining_lease` values were already missing, because the older files don't have that column at all.

The childcare file is different. It uses the text `na` for missing values, so this step matters there:

In [10]:
cc_raw = io_utils.read_csv_text(io_utils.find_files(FOLDERS, ChildcareCleaner.patterns)[0])
print('centre_name written as "na" BEFORE:', (cc_raw["centre_name"].str.lower() == "na").sum(),
      "| real missing:", cc_raw["centre_name"].isna().sum())
cc_marked = steps.mark_missing(cc_raw.copy())
print('centre_name written as "na" AFTER: ', (cc_marked["centre_name"].str.lower() == "na").sum(),
      "| real missing:", cc_marked["centre_name"].isna().sum())

centre_name written as "na" BEFORE: 67 | real missing: 0
centre_name written as "na" AFTER:  0 | real missing: 67


### 3c. Drop columns that are completely empty

In [11]:
df, empty = steps.drop_empty_columns(df)
print("dropped:", empty or "nothing")

dropped: nothing


### 3d. Convert numbers stored as text into real numbers (IDs stay text)

In [12]:
show_code(steps.is_id)
print("columns treated as IDs (kept as text):", config.ID_WORDS)

```python
def is_id(col: str) -> bool:
    return any(word in col.lower() for word in config.ID_WORDS)

```

columns treated as IDs (kept as text): ('postal', 'contact', 'phone', 'block', 'blk', 'code', 'objectid', 'unique_id', 'bus_stop_num', 'bus_roof_num', 'crc')


In [13]:
types_before = df.dtypes.astype(str)
df = steps.convert_types(df, skip=ResaleCleaner.skip_type_conversion)
pd.DataFrame({"before": types_before, "after": df.dtypes.astype(str)})

,before,after
month,str,str
town,str,str
flat_type,str,str
block,str,str
street_name,str,str
storey_range,str,str
floor_area_sqm,str,float64
flat_model,str,str
lease_commence_date,str,Int64
resale_price,str,float64


### 3e. Duplicates: flag, don't delete

Two different flats in the same block, the same storey range, the same size and the same price in the same month look identical, but they are **real separate sales**. So we keep them and add two columns:
- `dup_count`: how many identical rows exist in the group
- `is_duplicate`: True for the 2nd, 3rd ... copy

In [14]:
show_code(steps.handle_duplicates)

```python
def handle_duplicates(df: pd.DataFrame, mode: str, subset=None) -> tuple[pd.DataFrame, int]:
    """mode = 'flag' (add columns, keep rows), 'drop' (delete repeats) or 'off'.

    'flag' adds:
      dup_count     how many identical rows exist in the group (1 = unique)
      is_duplicate  True for the 2nd, 3rd ... copy
    """
    if mode == "off":
        return df, 0
    cols = subset or list(df.columns)
    repeats = df.duplicated(subset=cols, keep="first")
    n = int(repeats.sum())
    if mode == "drop":
        return df[~repeats], n
    hashes = pd.util.hash_pandas_object(df[cols], index=False)
    df["dup_count"] = hashes.map(hashes.value_counts()).astype("Int64")
    df["is_duplicate"] = repeats
    return df, n

```

In [15]:
subset = [c for c in df.columns if c != "source_file"]
df, n = steps.handle_duplicates(df, "flag", subset)
print(f"{n:,} rows flagged as possible duplicates; rows kept: {len(df):,}")
df[df["dup_count"] > 1].sort_values(subset).head(6)

1,929 rows flagged as possible duplicates; rows kept: 987,234


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,source_file,remaining_lease,date_basis,dup_count,is_duplicate
672,1990-01,GEYLANG,3 ROOM,47,CIRCUIT RD,01 TO 03,56.0,STANDARD,1969,18000.0,"Resale Flat Prices (Based on Approval Date), 1...",NaN,approval,2,False
673,1990-01,GEYLANG,3 ROOM,47,CIRCUIT RD,01 TO 03,56.0,STANDARD,1969,18000.0,"Resale Flat Prices (Based on Approval Date), 1...",NaN,approval,2,True
725,1990-01,HOUGANG,3 ROOM,308,HOUGANG AVE 5,10 TO 12,67.0,NEW GENERATION,1983,47000.0,"Resale Flat Prices (Based on Approval Date), 1...",NaN,approval,2,False
726,1990-01,HOUGANG,3 ROOM,308,HOUGANG AVE 5,10 TO 12,67.0,NEW GENERATION,1983,47000.0,"Resale Flat Prices (Based on Approval Date), 1...",NaN,approval,2,True
842,1990-01,JURONG WEST,3 ROOM,145,HU CHING RD,04 TO 06,64.0,IMPROVED,1976,23400.0,"Resale Flat Prices (Based on Approval Date), 1...",NaN,approval,2,False
843,1990-01,JURONG WEST,3 ROOM,145,HU CHING RD,04 TO 06,64.0,IMPROVED,1976,23400.0,"Resale Flat Prices (Based on Approval Date), 1...",NaN,approval,2,True


### 3f. Fill missing numbers: OFF by default

Filling a missing value with the median *invents* data, so it is switched off unless a dataset explicitly sets `fill = "median"`. When it is on, a `<column>_was_filled` flag is added so you can always see what was invented.

In [16]:
df, n = steps.fill_numeric(df, ResaleCleaner.fill)
print("fill setting for resale:", ResaleCleaner.fill, "-> values filled:", n)

fill setting for resale: None -> values filled: 0


## 4. Resale-specific steps

These are in `ResaleCleaner.extra_steps()`. The class **inherits** the generic steps from `DatasetCleaner` and only adds what is special about resale data.

In [17]:
show_code(ResaleCleaner)

```python
class ResaleCleaner(DatasetCleaner):
    name = "resale_transactions"
    patterns = ("Resale*Approval*.csv", "Resale*Registration*.csv")
    required = True
    duplicates = "flag"
    skip_type_conversion = ("month", "storey_range", "remaining_lease")

    def read(self):
        df = super().read()
        df["date_basis"] = df["source_file"].str.lower().map(
            lambda n: "approval" if "approval" in n else "registration")
        return df

    def extra_steps(self, df):
        # --- consistent spellings ------------------------------------------
        df["flat_type"] = (df["flat_type"].str.upper()
                           .str.replace("MULTI GENERATION", "MULTI-GENERATION", regex=False))
        df["flat_model"] = _canonical_case(df["flat_model"])

        # --- dates ------------------------------------------------------------
        df["sale_date"] = pd.to_datetime(df["month"] + "-01", format="%Y-%m-%d")
        df["sale_year"] = df["sale_date"].dt.year
        df["sale_month"] = df["sale_date"].dt.month

        # --- storeys (raw text kept) --------------------------------------------
        st = df["storey_range"].str.extract(r"(\d+)\s+TO\s+(\d+)").astype("Int64")
        df["storey_min"], df["storey_max"] = st[0], st[1]
        df["storey_bucket_size"] = df["storey_max"] - df["storey_min"] + 1

        # --- remaining lease ------------------------------------------------------
        official = (df["remaining_lease"].map(_lease_to_months).astype("Int64")
                    if "remaining_lease" in df else pd.Series(pd.NA, index=df.index, dtype="Int64"))
        months_used = (df["sale_year"] - df["lease_commence_date"].astype("Int64")) * 12 + (df["sale_month"] - 1)
        # sold before the lease officially started -> cap at the full 99 years
        estimate = (config.LEASE_YEARS * 12 - months_used).clip(upper=config.LEASE_YEARS * 12).astype("Int64")
        df["remaining_lease_months"] = official.fillna(estimate)
        df["lease_is_estimated"] = official.isna()

        # sort oldest -> newest; keep a stable column order
        df = df.sort_values(["sale_date", "town", "block"], kind="stable")
        front = ["month", "sale_date", "sale_year", "sale_month", "date_basis", "town", "flat_type",
                 "block", "street_name", "storey_range", "storey_min", "storey_max", "storey_bucket_size",
                 "floor_area_sqm", "flat_model", "lease_commence_date", "remaining_lease",
                 "remaining_lease_months", "lease_is_estimated", "resale_price"]
        front = [c for c in front if c in df]
        return df[front + [c for c in df.columns if c not in front]]

    def save(self, df, out_dir):
        out = df.copy()
        out["sale_date"] = out["sale_date"].dt.strftime("%Y-%m-%d")
        super().save(out, out_dir)

```

In [18]:
print("flat_model spellings BEFORE:", sorted(df["flat_model"].dropna().unique())[:12], "...")
print("flat_type values BEFORE:   ", sorted(df["flat_type"].dropna().unique()))
df = cleaner.extra_steps(df)
print()
print("flat_model spellings AFTER: ", sorted(df["flat_model"].dropna().unique())[:12], "...")
print("flat_type values AFTER:    ", sorted(df["flat_type"].dropna().unique()))

flat_model spellings BEFORE: ['2-ROOM', '2-room', '3Gen', 'APARTMENT', 'Adjoined flat', 'Apartment', 'DBSS', 'IMPROVED', 'IMPROVED-MAISONETTE', 'Improved', 'Improved-Maisonette', 'MAISONETTE'] ...


flat_type values BEFORE:    ['1 ROOM', '2 ROOM', '3 ROOM', '4 ROOM', '5 ROOM', 'EXECUTIVE', 'MULTI GENERATION', 'MULTI-GENERATION']



flat_model spellings AFTER:  ['2-room', '3Gen', 'Adjoined flat', 'Apartment', 'DBSS', 'Improved', 'Improved-Maisonette', 'Maisonette', 'Model A', 'Model A-Maisonette', 'Model A2', 'Multi Generation'] ...
flat_type values AFTER:     ['1 ROOM', '2 ROOM', '3 ROOM', '4 ROOM', '5 ROOM', 'EXECUTIVE', 'MULTI-GENERATION']


In [19]:
# New columns: dates split, storeys split (raw text kept), lease official vs estimated
cols = ["month", "sale_date", "date_basis", "storey_range", "storey_min", "storey_max", "storey_bucket_size",
        "lease_commence_date", "remaining_lease", "remaining_lease_months", "lease_is_estimated"]
pd.concat([df[cols].head(3), df[cols].tail(3)])

,month,sale_date,date_basis,storey_range,storey_min,storey_max,storey_bucket_size,lease_commence_date,remaining_lease,remaining_lease_months,lease_is_estimated
150,1990-01,1990-01-01,approval,07 TO 09,7,9,3,1978,NaN,1044,True
109,1990-01,1990-01-01,approval,07 TO 09,7,9,3,1978,NaN,1044,True
110,1990-01,1990-01-01,approval,04 TO 06,4,6,3,1978,NaN,1044,True
986975,2026-09,2026-09-01,registration,01 TO 03,1,3,3,1988,60 years 10 months,730,False
986976,2026-09,2026-09-01,registration,10 TO 12,10,12,3,1988,60 years 05 months,725,False
987057,2026-09,2026-09-01,registration,07 TO 09,7,9,3,1993,65 years 06 months,786,False


In [20]:
# Two storey-range schemes exist in the data: 3-floor and 5-floor buckets
df.groupby("date_basis")["storey_bucket_size"].value_counts().unstack(fill_value=0)

storey_bucket_size,3,5
date_basis,,
approval,656847,0
registration,323549,6838


In [21]:
# Remaining lease: how many rows use the official value vs our estimate, per period
df.groupby(df["sale_year"] // 5 * 5)["lease_is_estimated"].mean().rename("share estimated").to_frame()

,share estimated
sale_year,
1990,1.0
1995,1.0
2000,1.0
2005,1.0
2010,1.0
2015,0.0
2020,0.0
2025,0.0


## 5. Other datasets

Each one is a small class. The pattern is the same: read, generic steps, extra steps, save.

### Childcare centres

In [22]:
show_code(ChildcareCleaner.extra_steps)

```python
    def extra_steps(self, df):
        if "postal_code" in df:
            df["postal_code"] = df["postal_code"].str.zfill(6)

        df["name_missing"] = df["centre_name"].isna()
        df["centre_key"] = df["centre_code"].fillna(df.get("tp_code"))

        for col in [c for c in df.columns if "vacancy" in c]:
            df[f"{col}_code"] = df[col].map(config.VACANCY_CODES).astype("Int64")

        identifiable = df["centre_key"].notna() | df["centre_address"].notna()
        self.quarantine = df[~identifiable]
        self.report["quarantined"] = len(self.quarantine)
        self.report["name_missing_kept"] = int(df.loc[identifiable, "name_missing"].sum())
        return df[identifiable]

```

In [23]:
cc = ChildcareCleaner(io_utils.find_files(FOLDERS, ChildcareCleaner.patterns))
childcare = cc.run(OUT / config.CLEANED_DIR)
print(cc.report)
# centres with no name are KEPT: they still have a key (tp_code) and an address
childcare.loc[childcare["name_missing"], ["centre_key", "centre_name", "centre_address", "organisation_description"]].head()

{'dataset': 'childcare_centres', 'files': 1, 'rows_in': 1870, 'empty_columns_dropped': 1, 'duplicate_mode': 'flag', 'duplicates_found': 0, 'values_filled': 0, 'quarantined': 0, 'name_missing_kept': 67, 'rows_out': 1870}


,centre_key,centre_name,centre_address,organisation_description
1793,TP1466,NaN,"23,Bukit Batok East Avenue 4,#04-01,Bukit Bato...",Private Operators
1797,TP1460,NaN,"518,Jurong West Street 52,#01-129,Jurong West,...",Private Operators
1798,TP1459,NaN,"119,EDGEFIELD PLAINS,#05-01,Punggol,820119",Private Operators
1802,TP1457,NaN,"1,Punggol Drive,#03-03,One Punggol,828629",Private Operators
1805,TP1434,NaN,"992B,BUANGKOK LINK,#01-181,BUANGKOK SQUARE,532992",Private Operators


In [24]:
# vacancy text keeps its value; a numeric code is added next to it
pd.crosstab(childcare["infant_vacancy_current_month"].fillna("(blank)"),
            childcare["infant_vacancy_current_month_code"].astype("string").fillna("-"))

infant_vacancy_current_month_code,-,0,1,2
infant_vacancy_current_month,,,,
Available,0,0,0,374
Full,0,306,0,0
Limited,0,0,213,0
Not Applicable,977,0,0,0


### MRT exits: station + exit code is not unique (MRT and LRT share names)

In [25]:
mc = MrtExitCleaner(io_utils.find_files(FOLDERS, MrtExitCleaner.patterns))
mrt = mc.run(OUT / config.CLEANED_DIR)
mrt.loc[mrt["station_exit_shared"], ["objectid", "station_na", "exit_code", "longitude", "latitude"]].sort_values("station_na")

,objectid,station_na,exit_code,longitude,latitude
128,21537,BEDOK NORTH MRT STATION,Exit B,103.919063,1.334166
133,21542,BEDOK NORTH MRT STATION,Exit B,103.919134,1.334372
45,21449,BUKIT PANJANG MRT STATION,Exit A,103.762034,1.378766
236,21640,BUKIT PANJANG MRT STATION,Exit A,103.761923,1.379008
305,21185,BUKIT PANJANG MRT STATION,Exit A,103.759598,1.380745
35,21439,CHOA CHU KANG MRT STATION,Exit D,103.744224,1.385705
100,21504,CHOA CHU KANG MRT STATION,Exit C,103.744518,1.385736
101,21505,CHOA CHU KANG MRT STATION,Exit A,103.744557,1.385431
496,21376,CHOA CHU KANG MRT STATION,Exit D,103.744110,1.385370
495,21375,CHOA CHU KANG MRT STATION,Exit C,103.744470,1.385730


### Parks: polygons get a proper centroid (the real shape is kept for MongoDB)

In [26]:
pc = ParkCleaner(io_utils.find_files(FOLDERS, ParkCleaner.patterns))
parks = pc.run(OUT / config.CLEANED_DIR)
parks[["name", "longitude", "latitude", "outside_sg"]].head()

,name,longitude,latitude,outside_sg
0,TAI KENG GARDENS PG,103.884926,1.346185,False
1,TRAS LINK PK,103.843780,1.276456,False
2,GENTLE DR PG,103.840696,1.320368,False
3,FABER HEIGHTS PARK,103.755280,1.323879,False
4,KOON SENG PARK,103.902836,1.311048,False


## 6. Run the whole pipeline in one go

Everything above, for all datasets, plus:
- **tables**: shapes the cleaned data into the 6 MySQL tables
- **validate**: runs the quality checks
- **relationships**: proves the 1:M, M:1 and M:N relationships from the data

This takes about a minute.

In [27]:
from hdb_pipeline.run import run_pipeline
validation = run_pipeline(RAW, OUT, "clean,tables,validate,relationships")

== clean
  cleaning resale_transactions from 5 file(s)


  cleaning hdb_property_info from 1 file(s)


  cleaning childcare_centres from 1 file(s)


  cleaning hawker_centres from 1 file(s)
  cleaning bus_stops from 1 file(s)


  cleaning mrt_exits from 1 file(s)
  cleaning parks from 1 file(s)


  (skipped street_coordinates: no matching file)
  (not used: BusStopChineseNames.csv)
  (not used: CommuterFacilities.csv)
  (not used: NumberofMRTandLRTStations.csv)
            dataset  files  rows_in  empty_columns_dropped duplicate_mode  duplicates_found  values_filled  rows_out quarantined name_missing_kept
resale_transactions      5   987234                      0           flag              1929              0    987234                              
  hdb_property_info      1    13357                      0           flag                 0              0     13357                              
  childcare_centres      1     1870                      1           flag                 0              0      1870         0.0              67.0
     hawker_centres      1      129                      0           flag                 0              0       129                              
          bus_stops      1     5205                      0           flag                 0      

  town                        28 rows
  flat_type                    9 rows
  flat_model                  21 rows
  block                   13,549 rows
  block_unit_count        20,934 rows
  resale_transaction     987,234 rows
== validate
                                      check  failing_rows status                                            note
                   town: primary key unique             0   PASS                                                
              flat_type: primary key unique             0   PASS                                                
             flat_model: primary key unique             0   PASS                                                
                  block: primary key unique             0   PASS                                                
     resale_transaction: primary key unique             0   PASS                                                
       block_unit_count: primary key unique             0   PASS                  

                           relationship                  A                  B database kind  links  max_B_per_A  avg_B_per_A  max_A_per_B  avg_A_per_B expected matches_design                                                                        implemented_by
                        Town has Blocks               town              block      SQL  1:M  13549         1089        483.9            1          1.0      1:M            YES                                                              block.town_id FK -> town
          Block has Resale transactions              block resale_transaction      SQL  1:M 987234          849         98.6            1          1.0      1:M            YES                                               resale_transaction.block_id FK -> block
 Resale transactions are of a Flat type resale_transaction          flat_type      SQL  M:1 987234            1          1.0       378460     141033.4      M:1            YES                                       resa

## 7. Read the reports

In [28]:
reports = OUT / config.REPORTS_DIR
pd.read_csv(reports / "cleaning_report.csv")

,dataset,files,rows_in,empty_columns_dropped,duplicate_mode,duplicates_found,values_filled,rows_out,quarantined,name_missing_kept
0,resale_transactions,5,987234,0,flag,1929,0,987234,NaN,NaN
1,hdb_property_info,1,13357,0,flag,0,0,13357,NaN,NaN
2,childcare_centres,1,1870,1,flag,0,0,1870,0.0,67.0
3,hawker_centres,1,129,0,flag,0,0,129,NaN,NaN
4,bus_stops,1,5205,0,flag,0,0,5205,NaN,NaN
5,mrt_exits,1,613,0,flag,0,0,613,NaN,NaN
6,parks,1,462,0,flag,0,0,462,NaN,NaN


In [29]:
pd.read_csv(reports / "datasets_not_used.csv")

,file,reason
0,BusStopChineseNames.csv,no cleaner in datasets/REGISTRY (not needed by...
1,CommuterFacilities.csv,no cleaner in datasets/REGISTRY (not needed by...
2,NumberofMRTandLRTStations.csv,no cleaner in datasets/REGISTRY (not needed by...


In [30]:
v = pd.read_csv(reports / "validation_report.csv")
print(v["status"].value_counts().to_string())
v

status
PASS    25
WARN     7


,check,failing_rows,status,note
0,town: primary key unique,0,PASS,NaN
1,flat_type: primary key unique,0,PASS,NaN
2,flat_model: primary key unique,0,PASS,NaN
3,block: primary key unique,0,PASS,NaN
4,resale_transaction: primary key unique,0,PASS,NaN
5,block_unit_count: primary key unique,0,PASS,NaN
6,"block: (blk_no, street_name) unique",0,PASS,NaN
7,resale -> block,0,PASS,NaN
8,resale -> flat_type,0,PASS,NaN
9,resale -> flat_model,0,PASS,NaN


In [31]:
pd.read_csv(reports / "relationship_report.csv")[
    ["relationship", "database", "kind", "max_B_per_A", "max_A_per_B", "matches_design", "implemented_by"]]

,relationship,database,kind,max_B_per_A,max_A_per_B,matches_design,implemented_by
0,Town has Blocks,SQL,1:M,1089,1,YES,block.town_id FK -> town
1,Block has Resale transactions,SQL,1:M,849,1,YES,resale_transaction.block_id FK -> block
2,Resale transactions are of a Flat type,SQL,M:1,1,378460,YES,resale_transaction.flat_type_id FK -> flat_type
3,Resale transactions are of a Flat model,SQL,M:1,1,289041,YES,resale_transaction.flat_model_id FK -> flat_model
4,Blocks contain Flat types,SQL,M:N,5,8153,YES,"junction table block_unit_count (block_id, fla..."
5,Flat types come in Flat models,SQL,M:N,12,5,YES,derived through resale_transaction (both FKs o...
6,MRT station has Exits,MongoDB,1:M,13,1,YES,station name repeated on each exit document (o...
7,Operator type has Childcare centres,MongoDB,1:M,995,1,YES,organisation fields stored on each centre docu...


In [32]:
# The MySQL-ready tables
for name in ["town", "flat_type", "flat_model", "block", "block_unit_count", "resale_transaction"]:
    t = pd.read_csv(OUT / config.TABLES_DIR / f"{name}.csv", nrows=3)
    print(f"\n== {name}")
    display(t)


== town


,town_id,town_name,town_code
0,1,ANG MO KIO,AMK
1,2,BEDOK,BD
2,3,BISHAN,BH



== flat_type


,flat_type_id,flat_type_name
0,1,1 ROOM
1,2,2 ROOM
2,3,3 ROOM



== flat_model


,flat_model_id,model_name
0,1,2-room
1,2,3Gen
2,3,Adjoined flat


... (2 outputs left out)


== block_unit_count


,block_id,flat_type_id,tenure,unit_count
0,2,4,SOLD,29
1,2,5,SOLD,57
2,3,4,SOLD,29



== resale_transaction


,transaction_id,block_id,flat_type_id,flat_model_id,sale_date,date_basis,storey_range,storey_min,storey_max,floor_area_sqm,lease_commence_year,remaining_lease_months,lease_is_estimated,resale_price,dup_count,is_duplicate
0,1,443,5,18,1990-01-01,approval,07 TO 09,7,9,117,1978,1044,True,118000,1,False
1,2,445,4,13,1990-01-01,approval,07 TO 09,7,9,90,1978,1044,True,64500,1,False
2,3,523,4,13,1990-01-01,approval,04 TO 06,4,6,92,1978,1044,True,72500,1,False


## 8. (Optional) Geocode block coordinates

This looks up every block on OneMap, about 13,500 requests at a safe speed, so it takes **about 1.5 hours** and needs internet. Progress is saved every 100 blocks. Stop it any time with **Kernel → Interrupt**, and rerunning the cell continues where it stopped.

Set `RUN_GEOCODE = True` to start it. Afterwards the cell rebuilds the tables so the coordinates appear in `block`.

In [33]:
RUN_GEOCODE = False

if RUN_GEOCODE:
    run_pipeline(RAW, OUT, "geocode,tables,validate,relationships")
else:
    cache = OUT / config.GEOCODE_CACHE
    print("Geocoding skipped.", f"Cache has {len(pd.read_csv(cache))} blocks." if cache.exists() else "No cache yet.")

Geocoding skipped. No cache yet.
